In [38]:
import duckdb
from pathlib import Path

# Persist this time — Silver tables must survive the session so the next
# notebook and the invariant tests can read them.
DB = Path("../data/urbanflow.duckdb")
con = duckdb.connect(str(DB))

RAW = "../data/raw/yellow_tripdata_2024-*.parquet"

In [39]:
con.sql(f"""
CREATE OR REPLACE VIEW bronze_typed AS
SELECT
    md5(concat_ws('|',
        COALESCE(CAST(tpep_pickup_datetime  AS VARCHAR), ''),
        COALESCE(CAST(tpep_dropoff_datetime AS VARCHAR), ''),
        COALESCE(CAST(PULocationID AS VARCHAR), ''),
        COALESCE(CAST(DOLocationID AS VARCHAR), ''),
        COALESCE(CAST(trip_distance AS VARCHAR), ''),
        COALESCE(CAST(fare_amount   AS VARCHAR), ''),
        COALESCE(CAST(total_amount  AS VARCHAR), ''),
        COALESCE(CAST(payment_type  AS VARCHAR), ''),
        COALESCE(CAST(VendorID      AS VARCHAR), '')
    )) AS trip_id,

    CAST(VendorID AS INTEGER)                        AS vendor_id,
    tpep_pickup_datetime                             AS pickup_ts,
    tpep_dropoff_datetime                            AS dropoff_ts,
    CAST(tpep_pickup_datetime AS DATE)               AS partition_date,

    -- keep the raw value so flags read the source, not the cleaned column
    passenger_count                                  AS passenger_count_raw,

    -- 0 passengers means "not entered", not zero people
    CASE WHEN passenger_count = 0 THEN NULL
         ELSE CAST(passenger_count AS INTEGER) END   AS passenger_count,

    CAST(trip_distance AS DECIMAL(10,2))             AS trip_distance,
    CAST(RatecodeID AS INTEGER)                      AS ratecode_id,

    CASE WHEN store_and_fwd_flag = 'Y' THEN TRUE
         WHEN store_and_fwd_flag = 'N' THEN FALSE
         ELSE NULL END                               AS store_and_fwd,

    CAST(PULocationID AS INTEGER)                    AS pu_location_id,
    CAST(DOLocationID AS INTEGER)                    AS do_location_id,
    CAST(payment_type AS INTEGER)                    AS payment_type,

    CAST(fare_amount   AS DECIMAL(10,2))             AS fare_amount,
    CAST(total_amount  AS DECIMAL(10,2))             AS total_amount,
    CAST(congestion_surcharge AS DECIMAL(10,2))      AS congestion_surcharge,
    CAST(Airport_fee   AS DECIMAL(10,2))             AS airport_fee,

    date_diff('minute', tpep_pickup_datetime, tpep_dropoff_datetime) AS trip_duration_min
FROM '{RAW}'
""")
con.sql("SELECT COUNT(*) AS rows FROM bronze_typed").df()

,rows
0,9554778


In [40]:
con.sql("""
CREATE OR REPLACE VIEW bronze_flagged AS
SELECT *,
    -- quarantine rules: routing key
    (dropoff_ts <= pickup_ts)          AS q_impossible_duration,
    (trip_duration_min > 720)          AS q_excessive_duration,
    (trip_distance > 200)              AS q_impossible_distance,

    -- keep-and-flag rules (read the raw passenger column, not the cleaned one)
    (passenger_count_raw IS NULL)      AS f_null_block,
    (passenger_count_raw = 0)          AS f_zero_passengers,
    (fare_amount < 0)                  AS f_negative_fare,
    (total_amount < 0)                 AS f_negative_total,
    (trip_distance = 0)                AS f_zero_distance,
    (vendor_id NOT IN (1,2))           AS f_unknown_vendor,
    (partition_date < DATE '2024-01-01'
        OR partition_date >= DATE '2024-04-01') AS f_out_of_period
FROM bronze_typed
""")

con.sql("""
SELECT
  SUM(q_impossible_duration::INT) AS impossible_duration,
  SUM(q_excessive_duration::INT)  AS excessive_duration,
  SUM(q_impossible_distance::INT) AS impossible_distance,
  SUM((q_impossible_duration OR q_excessive_duration OR q_impossible_distance)::INT) AS total_quarantined
FROM bronze_flagged
""").df()

,impossible_duration,excessive_duration,impossible_distance,total_quarantined
0,2801.0,4916.0,170.0,7887.0


In [41]:
# Quarantine table: physically impossible rows, with the failing rule attached
con.sql("""
CREATE OR REPLACE TABLE silver_trip_quarantine AS
SELECT
    trip_id, vendor_id, pickup_ts, dropoff_ts, partition_date,
    trip_distance, trip_duration_min, fare_amount, total_amount,
    pu_location_id, do_location_id,
    list_filter([
        CASE WHEN q_impossible_duration THEN 'IMPOSSIBLE_DURATION' END,
        CASE WHEN q_excessive_duration  THEN 'EXCESSIVE_DURATION'  END,
        CASE WHEN q_impossible_distance THEN 'IMPOSSIBLE_DISTANCE' END
    ], x -> x IS NOT NULL) AS quarantine_flags
FROM bronze_flagged
WHERE q_impossible_duration OR q_excessive_duration OR q_impossible_distance
""")

# Clean table: everything possible, with keep-flags recorded.
# Exact duplicates are collapsed to one row per trip_id (see ADR-004).
con.sql("""
CREATE OR REPLACE TABLE silver_trip AS
SELECT
    trip_id, vendor_id, pickup_ts, dropoff_ts, partition_date,
    passenger_count, trip_distance, ratecode_id, store_and_fwd,
    pu_location_id, do_location_id, payment_type,
    fare_amount, total_amount, congestion_surcharge, airport_fee,
    trip_duration_min,
    (total_amount > 0 AND trip_distance > 0) AS is_billable,
    list_filter([
        CASE WHEN f_null_block      THEN 'NULL_BLOCK'      END,
        CASE WHEN f_negative_fare   THEN 'NEGATIVE_FARE'   END,
        CASE WHEN f_negative_total  THEN 'NEGATIVE_TOTAL'  END,
        CASE WHEN f_zero_distance   THEN 'ZERO_DISTANCE'   END,
        CASE WHEN f_zero_passengers THEN 'ZERO_PASSENGERS' END,
        CASE WHEN f_unknown_vendor  THEN 'UNKNOWN_VENDOR'  END,
        CASE WHEN f_out_of_period   THEN 'OUT_OF_PERIOD'   END
    ], x -> x IS NOT NULL) AS dq_flags
FROM bronze_flagged
WHERE NOT (q_impossible_duration OR q_excessive_duration OR q_impossible_distance)
QUALIFY ROW_NUMBER() OVER (PARTITION BY trip_id ORDER BY pickup_ts) = 1
""")

con.sql("""
SELECT
  (SELECT COUNT(*) FROM silver_trip)            AS clean,
  (SELECT COUNT(*) FROM silver_trip_quarantine) AS quarantined,
  (SELECT COUNT(*) FROM silver_trip)
  + (SELECT COUNT(*) FROM silver_trip_quarantine) AS total
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clean,quarantined,total
0,9546890,7887,9554777


In [42]:
def check(name, sql, expect_zero=True):
    n = con.sql(sql).fetchone()[0]
    ok = (n == 0) if expect_zero else (n > 0)
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {n}")
    assert ok, f"INVARIANT FAILED: {name} = {n}"

src = con.sql("SELECT COUNT(*) FROM bronze_typed").fetchone()[0]

# how many rows were dropped as exact duplicates on trip_id (see ADR-004)
dupes_collapsed = con.sql("""
    SELECT COUNT(*) - COUNT(DISTINCT trip_id)
    FROM bronze_flagged
    WHERE NOT (q_impossible_duration OR q_excessive_duration OR q_impossible_distance)
""").fetchone()[0]
print(f"      (duplicates collapsed: {dupes_collapsed})")

# 1 — conservation: silver + quarantine + collapsed duplicates = source
check("1 conservation",
      f"""SELECT ABS({src}
          - (SELECT COUNT(*) FROM silver_trip)
          - (SELECT COUNT(*) FROM silver_trip_quarantine)
          - {dupes_collapsed})""")

# 2 — no trip_id in both tables
check("2 no overlap",
      """SELECT COUNT(*) FROM silver_trip s
         JOIN silver_trip_quarantine q USING (trip_id)""")

# 3 — every quarantined row has at least one quarantine flag
check("3 quarantine has reason",
      "SELECT COUNT(*) FROM silver_trip_quarantine WHERE len(quarantine_flags) = 0")

# 4 — dq_flags never NULL on silver_trip
check("4 flags not null",
      "SELECT COUNT(*) FROM silver_trip WHERE dq_flags IS NULL")

# 5 — no silver_trip row breaks a quarantine rule
check("5 no impossible in clean",
      """SELECT COUNT(*) FROM silver_trip
         WHERE dropoff_ts <= pickup_ts
            OR trip_duration_min > 720
            OR trip_distance > 200""")

# 6 — partition_date always derives from pickup
check("6 partition matches pickup",
      "SELECT COUNT(*) FROM silver_trip WHERE partition_date <> CAST(pickup_ts AS DATE)")

# 7 — trip_id is unique in silver_trip
check("7 trip_id unique",
      "SELECT COUNT(*) - COUNT(DISTINCT trip_id) FROM silver_trip")

print("\nAll invariants passed.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      (duplicates collapsed: 1)
PASS  1 conservation: 0
PASS  2 no overlap: 0
PASS  3 quarantine has reason: 0
PASS  4 flags not null: 0
PASS  5 no impossible in clean: 0
PASS  6 partition matches pickup: 0
PASS  7 trip_id unique: 0

All invariants passed.


In [43]:
con.sql("""
CREATE OR REPLACE TABLE dq_results AS
SELECT
    current_date AS run_date,
    date_trunc('month', pickup_ts) AS partition_month,
    flag,
    COUNT(*) AS row_count
FROM silver_trip, UNNEST(dq_flags) AS t(flag)
GROUP BY 1, 2, 3
ORDER BY 2, 3
""")
con.sql("SELECT * FROM dq_results WHERE flag = 'NULL_BLOCK' ORDER BY partition_month").df()

,run_date,partition_month,flag,row_count
0,2026-07-29,2024-01-01,NULL_BLOCK,140030
1,2026-07-29,2024-02-01,NULL_BLOCK,185518
2,2026-07-29,2024-03-01,NULL_BLOCK,425849
